# Simulate a toolpath with `shellsim`

Define a wheel toolpath, write it as the JSON the solver reads, run the adaptive-cutback shell solver, and look at the formed sheet.
Requires the compiled solver at `bin/shell` (see the root README) and the `environment.yml` Python environment.
Run this notebook from the repository root or from `shellsim/examples/notebooks/`.

In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'shellsim').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from shellsim import toolpath, simulate

## 1. Toolpath → JSON
A zigzag pass has stroke length `lv_mm`, `n_strips` strips of `width_mm` at angle `alpha_deg`, growth strains `gtop`/`gbot` (top/bottom face) and orthotropy; `rotation_deg` turns the pass in the sheet plane.
`toolpath.sequence` takes the passes in order; passes accumulate in order and the example solves once, after the complete sequence (add a repeat count as a third tuple item). Hardening is the face-shared Voce law.
The JSON is in mm and degrees; the solver geometry arguments below are in metres. This is a two-pass crosshatch from the dome search (screening attempt 21).

In [ ]:
common = dict(width_mm=8.0, gtop=0.000288351, gbot=1.4588e-05, ortho_top=-0.398916, ortho_bottom=0.960306,
              profile={'mode': 'center_peak', 'top_end_ratio': 0.527535, 'top_profile_power': 2.0})
seq = toolpath.sequence([
    ('A', toolpath.zigzag(lv_mm=167.148, alpha_deg=2.0915, n_strips=26, **common)),
    ('B', toolpath.zigzag(lv_mm=151.991, alpha_deg=5.8756, n_strips=11, rotation_deg=90.0, **common)),
], hardening=toolpath.VOCE)
out = ROOT / 'run/notebook_example'
out.mkdir(parents=True, exist_ok=True)
toolpath.save(seq, out / 'sequence.json')
print((out / 'sequence.json').read_text()[:500], '...')

## 2. Run the solver
`simulate.run` uses the shared settings in `shellsim/configs/default_adaptive_cutback.json` and raises if the equilibrium does not reach the gradient tolerance.
`lx`, `ly` span the 200 x 200 mm sheet; `res=0.02` is a 4 mm mesh and took about 20 minutes on 4 threads (tested); finer meshes take longer.

In [ ]:
state = simulate.run(seq, out, lx=0.1, ly=0.1, res=0.02, h_total=0.00077, tolerance='1e-14', threads=4)
proof = state['proof']
print(proof['termination'], 'gradient', proof['gradient_norm'], 'energy', proof['energy'])

## 3. Inspect the formed sheet

In [ ]:
x, y, z = simulate.height(state)
faces = np.asarray(state['faces'], int)
fig, ax = plt.subplots(figsize=(5, 5))
t = ax.tricontourf(x * 1e3, y * 1e3, faces, z * 1e3, levels=21, cmap='RdBu_r')
ax.set_aspect('equal'); ax.set_xlabel('x (mm)'); ax.set_ylabel('y (mm)')
fig.colorbar(t, label='height z (mm)')
print('peak-to-peak height: %.3f mm' % ((z.max() - z.min()) * 1e3))